# Experiments — reproducibility notebook

Reproduces every table, headline number and plot from Chapter 5 of the thesis. The
experiment loops live in this notebook directly: each cell loads the per-session
Phase-A dumps from `fixtures/{sessions,user-sessions,agent-sessions}/<id>/phase-a.json`
and produces an in-memory dataframe, no CSV intermediate.

The notebook depends on the analysis module's runtime classpath (about 200 MB of jars
including Equinox, JDT, RefactoringMiner, PMD). The `libs/` directory is gitignored;
regenerate it whenever a dependency changes with:

```bash
./gradlew :analysis:notebookLibs
```


## Setup — classpath, imports, helpers

In [1]:
@file:DependsOn("libs/JavaEWAH-1.2.3.jar")
@file:DependsOn("libs/Saxon-HE-12.5.jar")
@file:DependsOn("libs/alpn-api-1.1.3.v20160715.jar")
@file:DependsOn("libs/analysis-0.0.1.jar")
@file:DependsOn("libs/animal-sniffer-annotations-1.9.jar")
@file:DependsOn("libs/annotations-23.0.0.jar")
@file:DependsOn("libs/antlr4-runtime-4.13.2.jar")
@file:DependsOn("libs/aopalliance-1.0.jar")
@file:DependsOn("libs/asm-9.8.jar")
@file:DependsOn("libs/cglib-2.2.1-v20090111.jar")
@file:DependsOn("libs/checker-qual-3.49.2.jar")
@file:DependsOn("libs/ck-0.7.0.jar")
@file:DependsOn("libs/classindex-3.13.jar")
@file:DependsOn("libs/commonmark-0.24.0.jar")
@file:DependsOn("libs/commonmark-ext-gfm-tables-0.24.0.jar")
@file:DependsOn("libs/commons-codec-1.17.0.jar")
@file:DependsOn("libs/commons-csv-1.8.jar")
@file:DependsOn("libs/commons-io-2.17.0.jar")
@file:DependsOn("libs/commons-lang3-3.20.0.jar")
@file:DependsOn("libs/commons-logging-1.2.jar")
@file:DependsOn("libs/commons-text-1.15.0.jar")
@file:DependsOn("libs/config-1.4.3.jar")
@file:DependsOn("libs/core-3.0.0.jar")
@file:DependsOn("libs/ecj-3.45.0.jar")
@file:DependsOn("libs/error_prone_annotations-2.37.0.jar")
@file:DependsOn("libs/failureaccess-1.0.1.jar")
@file:DependsOn("libs/fastutil-8.5.18.jar")
@file:DependsOn("libs/gen.jdt-3.0.0.jar")
@file:DependsOn("libs/gen.treesitter-ng-4.0.0-beta6.jar")
@file:DependsOn("libs/github-api-1.330.jar")
@file:DependsOn("libs/gson-2.13.0.jar")
@file:DependsOn("libs/guava-30.1.1-jre.jar")
@file:DependsOn("libs/guice-3.0.jar")
@file:DependsOn("libs/httpclient5-5.1.3.jar")
@file:DependsOn("libs/httpcore5-5.1.3.jar")
@file:DependsOn("libs/httpcore5-h2-5.1.3.jar")
@file:DependsOn("libs/icu4j-74.2.jar")
@file:DependsOn("libs/j2objc-annotations-1.3.jar")
@file:DependsOn("libs/jackson-annotations-2.20.jar")
@file:DependsOn("libs/jackson-core-2.20.0.jar")
@file:DependsOn("libs/jackson-databind-2.20.0.jar")
@file:DependsOn("libs/jansi-2.4.1.jar")
@file:DependsOn("libs/java-diff-utils-4.16.jar")
@file:DependsOn("libs/javassist-3.26.0-GA.jar")
@file:DependsOn("libs/javax.inject-1.jar")
@file:DependsOn("libs/javax.servlet-api-3.1.0.jar")
@file:DependsOn("libs/jcommander-3.0.jar")
@file:DependsOn("libs/jetty-client-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-http-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-io-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-security-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-server-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-servlet-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-util-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-util-ajax-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-webapp-9.4.48.v20220622.jar")
@file:DependsOn("libs/jetty-xml-9.4.48.v20220622.jar")
@file:DependsOn("libs/jgrapht-core-1.5.1.jar")
@file:DependsOn("libs/jheaps-0.13.jar")
@file:DependsOn("libs/jna-5.18.1.jar")
@file:DependsOn("libs/jna-platform-5.18.1.jar")
@file:DependsOn("libs/jsoup-1.22.1.jar")
@file:DependsOn("libs/jsr305-3.0.2.jar")
@file:DependsOn("libs/jtidy-r938.jar")
@file:DependsOn("libs/jul-to-slf4j-1.7.36.jar")
@file:DependsOn("libs/junit-4.8.2.jar")
@file:DependsOn("libs/jzlib-1.1.3.jar")
@file:DependsOn("libs/kotlin-compiler-embeddable-2.3.20.jar")
@file:DependsOn("libs/kotlin-daemon-embeddable-2.3.20.jar")
@file:DependsOn("libs/kotlin-reflect-2.0.21.jar")
@file:DependsOn("libs/kotlin-script-runtime-2.3.20.jar")
@file:DependsOn("libs/kotlin-stdlib-2.3.20.jar")
@file:DependsOn("libs/kotlin-stdlib-jdk7-1.9.10.jar")
@file:DependsOn("libs/kotlin-stdlib-jdk8-1.9.10.jar")
@file:DependsOn("libs/kotlinx-coroutines-core-jvm-1.9.0.jar")
@file:DependsOn("libs/kotlinx-io-bytestring-jvm-0.5.4.jar")
@file:DependsOn("libs/kotlinx-io-core-jvm-0.5.4.jar")
@file:DependsOn("libs/kotlinx-serialization-core-jvm-1.7.3.jar")
@file:DependsOn("libs/kotlinx-serialization-json-io-jvm-1.7.3.jar")
@file:DependsOn("libs/kotlinx-serialization-json-jvm-1.7.3.jar")
@file:DependsOn("libs/ktor-events-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-http-cio-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-http-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-io-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-network-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-serialization-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-serialization-kotlinx-json-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-serialization-kotlinx-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-server-content-negotiation-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-server-core-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-server-netty-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-utils-jvm-3.0.3.jar")
@file:DependsOn("libs/ktor-websockets-jvm-3.0.3.jar")
@file:DependsOn("libs/kxs-ts-gen-core-jvm-0.2.2.jar")
@file:DependsOn("libs/listenablefuture-9999.0-empty-to-avoid-conflict-with-guava.jar")
@file:DependsOn("libs/log4j-1.2.17.jar")
@file:DependsOn("libs/logback-classic-1.5.32.jar")
@file:DependsOn("libs/logback-core-1.5.32.jar")
@file:DependsOn("libs/netty-buffer-4.1.116.Final.jar")
@file:DependsOn("libs/netty-codec-4.1.116.Final.jar")
@file:DependsOn("libs/netty-codec-http-4.1.116.Final.jar")
@file:DependsOn("libs/netty-codec-http2-4.1.116.Final.jar")
@file:DependsOn("libs/netty-common-4.1.116.Final.jar")
@file:DependsOn("libs/netty-handler-4.1.116.Final.jar")
@file:DependsOn("libs/netty-resolver-4.1.116.Final.jar")
@file:DependsOn("libs/netty-transport-4.1.116.Final.jar")
@file:DependsOn("libs/netty-transport-classes-epoll-4.1.116.Final.jar")
@file:DependsOn("libs/netty-transport-classes-kqueue-4.1.116.Final.jar")
@file:DependsOn("libs/netty-transport-native-epoll-4.1.116.Final.jar")
@file:DependsOn("libs/netty-transport-native-kqueue-4.1.116.Final.jar")
@file:DependsOn("libs/netty-transport-native-unix-common-4.1.116.Final.jar")
@file:DependsOn("libs/nice-xml-messages-3.1.jar")
@file:DependsOn("libs/org.apache.felix.scr-2.2.12.jar")
@file:DependsOn("libs/org.eclipse.core.commands-3.12.500.jar")
@file:DependsOn("libs/org.eclipse.core.contenttype-3.9.800.jar")
@file:DependsOn("libs/org.eclipse.core.expressions-3.9.500.jar")
@file:DependsOn("libs/org.eclipse.core.filebuffers-3.8.400.jar")
@file:DependsOn("libs/org.eclipse.core.filesystem-1.11.400.jar")
@file:DependsOn("libs/org.eclipse.core.jobs-3.15.700.jar")
@file:DependsOn("libs/org.eclipse.core.resources-3.23.200.jar")
@file:DependsOn("libs/org.eclipse.core.runtime-3.34.200.jar")
@file:DependsOn("libs/org.eclipse.core.variables-3.6.500.jar")
@file:DependsOn("libs/org.eclipse.debug.core-3.23.0.jar")
@file:DependsOn("libs/org.eclipse.equinox.app-1.7.600.jar")
@file:DependsOn("libs/org.eclipse.equinox.common-3.20.300.jar")
@file:DependsOn("libs/org.eclipse.equinox.preferences-3.12.100.jar")
@file:DependsOn("libs/org.eclipse.equinox.registry-3.12.600.jar")
@file:DependsOn("libs/org.eclipse.jdt.core-3.45.0.jar")
@file:DependsOn("libs/org.eclipse.jdt.core.manipulation-1.23.0.jar")
@file:DependsOn("libs/org.eclipse.jdt.debug-3.23.0.jar")
@file:DependsOn("libs/org.eclipse.jdt.launching-3.23.300.jar")
@file:DependsOn("libs/org.eclipse.jdt.launching.macosx-3.6.300.jar")
@file:DependsOn("libs/org.eclipse.jface.text-3.25.0.jar")
@file:DependsOn("libs/org.eclipse.jgit-6.10.1.202505221210-r.jar")
@file:DependsOn("libs/org.eclipse.ltk.core.refactoring-3.15.0.jar")
@file:DependsOn("libs/org.eclipse.osgi-3.24.100.jar")
@file:DependsOn("libs/org.eclipse.search.core-3.16.400.jar")
@file:DependsOn("libs/org.eclipse.swt-3.130.0.jar")
@file:DependsOn("libs/org.eclipse.swt.cocoa.macosx.aarch64-3.130.0.jar")
@file:DependsOn("libs/org.eclipse.text-3.14.600.jar")
@file:DependsOn("libs/org.osgi.namespace.extender-1.0.1.jar")
@file:DependsOn("libs/org.osgi.service.component-1.5.1.jar")
@file:DependsOn("libs/org.osgi.service.prefs-1.1.2.jar")
@file:DependsOn("libs/org.osgi.util.function-1.2.0.jar")
@file:DependsOn("libs/org.osgi.util.promise-1.3.0.jar")
@file:DependsOn("libs/org.osgi.util.tracker-1.5.4.jar")
@file:DependsOn("libs/osgi.annotation-8.1.0.jar")
@file:DependsOn("libs/p4java-2025.2.2872146.jar")
@file:DependsOn("libs/pcollections-4.0.2.jar")
@file:DependsOn("libs/pmd-core-7.13.0.jar")
@file:DependsOn("libs/pmd-java-7.13.0.jar")
@file:DependsOn("libs/refactoring-miner-3.1.3.jar")
@file:DependsOn("libs/reflections-0.9.12.jar")
@file:DependsOn("libs/rendersnake-1.9.0.jar")
@file:DependsOn("libs/shared-0.0.1.jar")
@file:DependsOn("libs/simmetrics-core-3.2.3.jar")
@file:DependsOn("libs/slf4j-api-2.0.17.jar")
@file:DependsOn("libs/slf4j-log4j12-1.7.30.jar")
@file:DependsOn("libs/snakeyaml-2.0.jar")
@file:DependsOn("libs/spark-core-2.9.4.jar")
@file:DependsOn("libs/spring-aop-4.1.6.RELEASE.jar")
@file:DependsOn("libs/spring-beans-4.1.6.RELEASE.jar")
@file:DependsOn("libs/spring-context-4.1.6.RELEASE.jar")
@file:DependsOn("libs/spring-core-4.1.6.RELEASE.jar")
@file:DependsOn("libs/spring-expression-4.1.6.RELEASE.jar")
@file:DependsOn("libs/spring-web-4.1.6.RELEASE.jar")
@file:DependsOn("libs/spring-webmvc-4.1.6.RELEASE.jar")
@file:DependsOn("libs/swc4j-2.1.0.jar")
@file:DependsOn("libs/swc4j-linux-x86_64-2.1.0.jar")
@file:DependsOn("libs/swc4j-macos-arm64-2.1.0.jar")
@file:DependsOn("libs/tree-sitter-0.25.3.jar")
@file:DependsOn("libs/tree-sitter-c-0.23.2.jar")
@file:DependsOn("libs/tree-sitter-c-sharp-0.23.1.jar")
@file:DependsOn("libs/tree-sitter-cmake-0.4.1a.jar")
@file:DependsOn("libs/tree-sitter-cpp-0.23.4.jar")
@file:DependsOn("libs/tree-sitter-go-0.23.3.jar")
@file:DependsOn("libs/tree-sitter-java-0.23.4.jar")
@file:DependsOn("libs/tree-sitter-javascript-0.23.1.jar")
@file:DependsOn("libs/tree-sitter-kotlin-0.3.8.1.jar")
@file:DependsOn("libs/tree-sitter-ocaml-0.23.2.jar")
@file:DependsOn("libs/tree-sitter-php-0.23.11.jar")
@file:DependsOn("libs/tree-sitter-python-0.23.4.jar")
@file:DependsOn("libs/tree-sitter-r-main-a.jar")
@file:DependsOn("libs/tree-sitter-ruby-0.23.1.jar")
@file:DependsOn("libs/tree-sitter-rust-0.23.1.jar")
@file:DependsOn("libs/tree-sitter-swift-0.5.0.jar")
@file:DependsOn("libs/tree-sitter-typescript-0.23.2.jar")
@file:DependsOn("libs/websocket-api-9.4.48.v20220622.jar")
@file:DependsOn("libs/websocket-client-9.4.48.v20220622.jar")
@file:DependsOn("libs/websocket-common-9.4.48.v20220622.jar")
@file:DependsOn("libs/websocket-server-9.4.48.v20220622.jar")
@file:DependsOn("libs/websocket-servlet-9.4.48.v20220622.jar")
@file:DependsOn("libs/xmlresolver-5.2.2-data.jar")
@file:DependsOn("libs/xmlresolver-5.2.2.jar")

import com.github.ethanhosier.analysis.metrics.derived.CleanlinessWeights
import com.github.ethanhosier.analysis.metrics.derived.ProcessScoreWeights
import com.github.ethanhosier.analysis.metrics.derived.ScoringConfig
import com.github.ethanhosier.analysis.pipeline.AnalysisReport
import com.github.ethanhosier.analysis.pipeline.PhaseAResult
import com.github.ethanhosier.analysis.pipeline.ReportAssembler
import kotlinx.serialization.json.Json
import java.io.File
import kotlin.math.exp
import kotlin.math.ln
import kotlin.random.Random

In [2]:
%use dataframe
%use kandy

In [3]:
val readJson = Json { ignoreUnknownKeys = true; isLenient = true }
// Walk up from cwd until we find a directory containing `fixtures/sessions`. This
// makes the notebook robust to being launched either from `tool/` (the documented
// setup) or from `tool/fixtures/notebooks/` (nbconvert's default cwd).
val repoRoot: File = run {
    var d = File(".").canonicalFile
    while (d != null && !File(d, "fixtures/sessions").isDirectory) d = d.parentFile
    requireNotNull(d) { "could not find repo root (no ancestor contains fixtures/sessions)" }
}

// Phase-A dumps live per-session as `<root>/.../<session-dir>/phase-a.json`.
// Walk the tree to find every dump and use the session dir's path relative to
// `root` (with `/` → `-`) as the fixture id, so agent-sessions/<stack>/NN stays
// disambiguated across stacks while flat sessions/NNN or user-sessions/<name>-NN
// keep their natural ids.
fun loadCorpus(dir: String): Map<String, PhaseAResult> {
    val root = File(repoRoot, dir)
    return root.walk()
        .filter { it.isFile && it.name == "phase-a.json" }
        .map { it.parentFile }
        .sortedBy { it.relativeTo(root).path }
        .associate { sessionDir ->
            val sid = sessionDir.relativeTo(root).path.replace(File.separatorChar, '-')
            sid to readJson.decodeFromString(
                PhaseAResult.serializer(),
                File(sessionDir, "phase-a.json").readText(),
            )
        }
}

val injection = loadCorpus("fixtures/sessions")
val userStudy = loadCorpus("fixtures/user-sessions")
val agent     = loadCorpus("fixtures/agent-sessions")

println("loaded: injection=${injection.size}, user-study=${userStudy.size}, agent=${agent.size}")

loaded: injection=45, user-study=30, agent=48


In [4]:
// Helpers — Kendall τ-b and top-N hit rate over divergence-point rankings.
// Match RankingMetrics.kt in the analysis module (inlined here to keep
// the notebook self-contained for verification).

// Kendall τ-b: correlation between two rankings of the same items.
// −1 means fully inverted, 0 means independent, 1 means identical.
// Counts concordant vs discordant pairs across all i < j; the -b
// variant adjusts the denominator for ties so short rankings still
// score in [-1, 1].
fun kendallTauB(a: List<Int>, b: List<Int>): Double {
    if (a.size < 2 || b.size < 2) return if (a == b) 1.0 else Double.NaN
    val n = a.size.coerceAtMost(b.size)
    var concordant = 0; var discordant = 0; var tiesA = 0; var tiesB = 0
    for (i in 0 until n - 1) for (j in i + 1 until n) {
        val da = a[i].compareTo(a[j])
        val db = b[i].compareTo(b[j])
        when {
            da == 0 && db == 0 -> { tiesA++; tiesB++ }
            da == 0 -> tiesA++
            db == 0 -> tiesB++
            da * db > 0 -> concordant++
            else -> discordant++
        }
    }
    val denom = kotlin.math.sqrt(
        ((concordant + discordant + tiesA).toDouble()) *
        ((concordant + discordant + tiesB).toDouble())
    )
    return if (denom == 0.0) Double.NaN else (concordant - discordant) / denom
}

// Top-N hit rate: fraction of the baseline's top-N items that also
// appear in the perturbed top-N. 1.0 = the top-N set is preserved
// (internal order irrelevant). On a ranking shorter than N, both
// sides are taken as their full ranking, so the metric trivially
// returns 1.0 unless the underlying set actually changed.
fun topNHitRate(baseline: List<Int>, perturbed: List<Int>, n: Int): Double {
    if (baseline.isEmpty()) return 1.0
    val k = minOf(n, baseline.size)
    val a = baseline.take(k).toSet()
    val b = perturbed.take(k).toSet()
    return (a intersect b).size.toDouble() / k
}

// Ranking — the production ordering the two metrics above consume.
// Divergence points sorted by descending magnitude; ties broken by
// ascending stepIndex so the ranking is a deterministic function of
// the report. Ties matter on this corpus because COMMIT_GAP DPs all
// carry magnitude exactly W_cg (the commit-flip alt drops the gap
// count from 1 to 0 at trajectory-final), and saturated DPs share a
// clamp-induced magnitude.
fun rankingOf(report: AnalysisReport): List<Int> =
    report.divergencePoints
        .sortedWith(
            compareByDescending<com.github.ethanhosier.analysis.pipeline.DivergencePoint> { it.magnitude }
                .thenBy { it.stepIndex }
        )
        .map { it.stepIndex }

## §5.1 Testing the score formula

### §5.1.1 Single-knob sensitivity

For each (fixture, weight, factor) triple, scales one weight in ScoringConfig.PRODUCTION
by `factor`, reassembles the report via Phase B, and compares the perturbed divergence-point
ranking against production.

In [5]:
val FACTORS = listOf(0.1, 0.25, 0.5, 0.75, 1.25, 1.5, 2.0, 4.0, 10.0)

data class Knob(val name: String, val group: String, val perturb: (ScoringConfig, Double) -> ScoringConfig)

val KNOBS = listOf(
    Knob("gain", "process")        { c, f -> c.copy(process = c.process.copy(gain = c.process.gain * f)) },
    Knob("broken", "process")      { c, f -> c.copy(process = c.process.copy(broken = c.process.broken * f)) },
    Knob("skipTests", "process")   { c, f -> c.copy(process = c.process.copy(skipTests = c.process.skipTests * f)) },
    Knob("manualIde", "process")   { c, f -> c.copy(process = c.process.copy(manualIde = c.process.manualIde * f)) },
    Knob("length", "process")      { c, f -> c.copy(process = c.process.copy(length = c.process.length * f)) },
    Knob("commitGap", "process")   { c, f -> c.copy(process = c.process.copy(commitGap = c.process.commitGap * f)) },
    Knob("cognitive", "cleanliness")   { c, f -> c.copy(cleanliness = c.cleanliness.copy(cognitive = c.cleanliness.cognitive * f)) },
    Knob("coupling", "cleanliness")    { c, f -> c.copy(cleanliness = c.cleanliness.copy(coupling = c.cleanliness.coupling * f)) },
    Knob("duplication", "cleanliness") { c, f -> c.copy(cleanliness = c.cleanliness.copy(duplication = c.cleanliness.duplication * f)) },
    Knob("readability", "cleanliness") { c, f -> c.copy(cleanliness = c.cleanliness.copy(readability = c.cleanliness.readability * f)) },
    Knob("smells", "cleanliness")      { c, f -> c.copy(cleanliness = c.cleanliness.copy(smells = c.cleanliness.smells * f)) },
    Knob("cohesion", "cleanliness")    { c, f -> c.copy(cleanliness = c.cleanliness.copy(cohesion = c.cleanliness.cohesion * f)) },
)

// Saturation: a divergence point is "saturated" if any of its
// referenced alts' terminal process score or the matching user
// terminal score sits at the [0, 100] clamp boundary. Saturated DPs
// have weight-invariant magnitudes (the clamp eats any perturbation),
// so they inflate τ artificially when included; reported separately
// to bound the clipping-as-confound concern in §5.1.
fun saturatedDpCount(report: AnalysisReport): Int {
    val userProc = report.checkpoints.associate { it.sha to it.derivedMetrics.process.total }
    var n = 0
    for (dp in report.divergencePoints) {
        val touches = dp.altTrajectoryIndexes.mapNotNull { report.alternativeTrajectories.getOrNull(it) }.any { alt ->
            val a = alt.altCheckpoints.lastOrNull()?.derivedMetrics?.process?.total
            val u = userProc[alt.userToSha]
            (a != null && (a == 0 || a == 100)) || (u != null && (u == 0 || u == 100))
        }
        if (touches) n += 1
    }
    return n
}

fun sensitivitySweep(name: String, corpus: Map<String, PhaseAResult>) =
    corpus.flatMap { (fixture, phaseA) ->
        val baseline = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
        val baselineRanking = rankingOf(baseline)
        val baselineSat = saturatedDpCount(baseline)
        KNOBS.flatMap { knob ->
            FACTORS.map { factor ->
                val perturbed = ReportAssembler.assemble(phaseA, knob.perturb(ScoringConfig.PRODUCTION, factor))
                val pr = rankingOf(perturbed)
                mapOf(
                    "set" to name, "fixture" to fixture, "group" to knob.group, "weight" to knob.name, "factor" to factor,
                    "tau" to kendallTauB(baselineRanking, pr),
                    "top1" to topNHitRate(baselineRanking, pr, 1),
                    "top3" to topNHitRate(baselineRanking, pr, 3),
                    "top5" to topNHitRate(baselineRanking, pr, 5),
                    "baseline_size" to baselineRanking.size,
                    "baseline_sat" to baselineSat,
                )
            }
        }
    }.toDataFrame()

val sensInj    = sensitivitySweep("Injection (45)",  injection)
val sensUser   = sensitivitySweep("User study (12)", userStudy)
val sensAgent  = sensitivitySweep("Agent (6)",       agent)

Headline table — top-1 stability, τ = 1.0 share, and baseline saturation per session set (reproduces Table 5.1).

In [6]:
fun pct(p: Double) = "%.1f%%".format(p * 100)

val sensHeadline = listOf(sensInj, sensUser, sensAgent).map { df ->
    val n = df.rowsCount()
    val baselineDps = df["baseline_size"].cast<Int>().sum()
    mapOf(
        "session set" to (df["set"][0] as String),
        "rows" to n,
        "tau=1.0" to pct(df.filter { "tau"<Double>() == 1.0 }.rowsCount().toDouble() / n),
        "top-1=1" to pct(df.filter { "top1"<Double>() == 1.0 }.rowsCount().toDouble() / n),
        "tau<0"   to pct(df.filter { "tau"<Double>() < 0.0 }.rowsCount().toDouble() / n),
        "baseline saturation" to pct(df["baseline_sat"].cast<Int>().sum().toDouble() / baselineDps.coerceAtLeast(1)),
    )
}.toDataFrame()
sensHeadline

session set,rows,tau=1.0,top-1=1,tau<0,baseline saturation
Injection (45),4860,77.0%,99.8%,0.3%,10.6%
User study (12),3240,51.5%,96.9%,0.8%,6.4%
Agent (6),5184,88.5%,98.5%,1.2%,17.5%


Per-knob top-1 disruption counts on the user-study session set (reproduces Table 5.2).

In [7]:
val perKnob = sensUser
    .groupBy("weight")
    .aggregate {
        count { "top1"<Double>() < 1.0 } into "top1_disrupt"
        count() into "rows"
    }
    .sortByDesc("top1_disrupt")
perKnob

weight,top1_disrupt,rows
gain,24,270
manualIde,17,270
skipTests,15,270
commitGap,15,270
length,14,270
broken,12,270
coupling,2,270
smells,1,270
cohesion,1,270
cognitive,0,270


In [8]:
perKnob.plot {
    bars {
        x("weight"); y("top1_disrupt")
    }
    layout { title = "Per-knob top-1 disruption counts (user-study set)" }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="G87ZpR" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("G87ZpR");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Per-knob top-1 disruption counts (user-study set)"
},
"mapping":{
},
"data":{
"weight":["gain","manualIde","skipTests","commitGap","length","broken","coupling","smells","cohesion","cognitive","duplication","readability"],
"top1_disrupt":[24.0,17.0,15.0,15.0,14.0,12.0,2.0,1.0,1.0,0.0,0.0,0.0]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"weight",
"y":"top1_disrupt"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"weight"
},{
"type":"int",
"column":"top1_disrupt"
}]
},
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gain 
 
 
 
 
 
 
 
 
 manualIde 
 
 
 
 
 
 
 
 
 skipTests 
 
 
 
 
 
 
 
 
 commitGap 
 
 
 
 
 
 
 
 
 length 
 
 
 
 
 
 
 
 
 broken 
 
 
 
 
 
 
 
 
 coupling 
 
 
 
 
 
 
 
 
 smells 
 
 
 
 
 
 
 
 
 cohesion 
 
 
 
 
 
 
 
 
 cognitive 
 
 
 
 
 
 
 
 
 duplication 
 
 
 
 
 
 
 
 
 readability 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 25 
 
 
 
 
 
 
 
 
 Per-knob top-1 disruption counts (user-study set) 
 
 
 
 
 top1_disrupt 
 
 
 
 
 weight

### Multi-knob Monte Carlo

Every weight scaled by an independent log-normal $\exp(\sigma Z)$, $\sigma = \ln 2$, drawn
from a per-fixture seeded RNG. 200 samples per fixture; mean τ and top-N hit rate against
production. The headline dataframe at the end of this cell reproduces Table 5.3.

In [9]:
fun nextGaussian(rng: Random): Double {
    var u: Double; var v: Double; var s: Double
    do {
        u = rng.nextDouble(-1.0, 1.0); v = rng.nextDouble(-1.0, 1.0); s = u * u + v * v
    } while (s >= 1.0 || s == 0.0)
    return u * kotlin.math.sqrt(-2.0 * ln(s) / s)
}

// Multi-knob perturbation: every weight gets an independent
// multiplicative factor drawn from exp(σ · Z), Z ~ N(0, 1). With
// σ = ln 2 the factor lands inside [×0.25, ×4] for ~95% of samples
// and the distribution is symmetric in log-space, so up-scalings and
// down-scalings get the same weight.
fun perturbAll(rng: Random, sigma: Double): ScoringConfig {
    fun f() = exp(sigma * nextGaussian(rng))
    val p = ScoringConfig.PRODUCTION
    return p.copy(
        process = ProcessScoreWeights(
            gain = p.process.gain * f(), broken = p.process.broken * f(),
            skipTests = p.process.skipTests * f(), manualIde = p.process.manualIde * f(),
            length = p.process.length * f(), commitGap = p.process.commitGap * f(),
        ),
        cleanliness = CleanlinessWeights(
            cognitive = p.cleanliness.cognitive * f(), coupling = p.cleanliness.coupling * f(),
            duplication = p.cleanliness.duplication * f(), readability = p.cleanliness.readability * f(),
            smells = p.cleanliness.smells * f(), cohesion = p.cleanliness.cohesion * f(),
        ),
    )
}

fun mcSweep(name: String, corpus: Map<String, PhaseAResult>, samples: Int = 200, seed: Long = 1L) =
    corpus.entries.withIndex().flatMap { (idx, entry) ->
        val (fixture, phaseA) = entry
        val baseline = rankingOf(ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION))
        val rng = Random(seed + idx * 1_000_003L)
        (0 until samples).map { s ->
            val cfg = perturbAll(rng, ln(2.0))
            val pr = rankingOf(ReportAssembler.assemble(phaseA, cfg))
            mapOf(
                "set" to name, "fixture" to fixture, "sample" to s,
                "tau" to kendallTauB(baseline, pr),
                "top1" to topNHitRate(baseline, pr, 1),
                "top3" to topNHitRate(baseline, pr, 3),
                "top5" to topNHitRate(baseline, pr, 5),
                "baseline_size" to baseline.size,
            )
        }
    }.toDataFrame()

val mcInj   = mcSweep("Injection (45)",  injection)
val mcUser  = mcSweep("User study (12)", userStudy)
val mcAgent = mcSweep("Agent (6)",       agent)

val mcHeadline = listOf(mcInj, mcUser, mcAgent).map { df ->
    val n = df.rowsCount()
    mapOf(
        "session set" to (df["set"][0] as String),
        "samples" to n,
        "mean tau" to "%.3f".format(df["tau"].cast<Double>().mean()),
        "top-1=1" to pct(df.filter { "top1"<Double>() == 1.0 }.rowsCount().toDouble() / n),
        "top-3=1" to pct(df.filter { "top3"<Double>() == 1.0 }.rowsCount().toDouble() / n),
        "top-5=1" to pct(df.filter { "top5"<Double>() == 1.0 }.rowsCount().toDouble() / n),
    )
}.toDataFrame()
mcHeadline

session set,samples,mean tau,top-1=1,top-3=1,top-5=1
Injection (45),9000,0.805,99.1%,77.8%,77.8%
User study (12),6000,0.656,86.9%,68.0%,66.6%
Agent (6),9600,0.848,92.6%,91.0%,89.6%


In [10]:
// τ distribution per session set — three histograms in series, one per set.
// (Faceting / per-aesthetic colouring varies across Kandy versions; emitting
// one plot per call is the version-robust choice.)
listOf(mcInj, mcUser, mcAgent).forEach { df ->
    val setName = df["set"][0] as String
    df.plot {
        histogram("tau")
        layout { title = "τ distribution — $setName" }
    }
}

### §5.1.2 Ablation

Power-set sweep over the six process-side weights ($2^6 = 64$ variants per fixture).
Cleanliness weights stay at production across all variants because the sensitivity sweep
already established they don't shift the ranking.

In [11]:
val PROC_TERMS = listOf("gain", "broken", "skipTests", "manualIde", "length", "commitGap", "lag")
val allTerms = PROC_TERMS.toSortedSet()

// Ablation: build a ScoringConfig in which the named process-side
// weights keep their production values and every weight outside the
// set is zeroed. Cleanliness weights stay at production across all
// variants (sensitivity sweep showed they don't move the ranking).
// Iterating over the 2^6 power-set of process weights asks which
// subsets recover the production ranking under Kendall τ.
fun ablate(active: Set<String>): ScoringConfig {
    val p = ScoringConfig.PRODUCTION.process
    return ScoringConfig.PRODUCTION.copy(
        process = ProcessScoreWeights(
            gain      = if ("gain"      in active) p.gain      else 0.0,
            broken    = if ("broken"    in active) p.broken    else 0.0,
            skipTests = if ("skipTests" in active) p.skipTests else 0.0,
            manualIde = if ("manualIde" in active) p.manualIde else 0.0,
            length    = if ("length"    in active) p.length    else 0.0,
            commitGap = if ("commitGap" in active) p.commitGap else 0.0,
            lag       = if ("lag"       in active) p.lag       else 0.0,
        )
    )
}

fun powerSet(items: List<String>): List<Set<String>> {
    val out = mutableListOf<Set<String>>()
    val n = items.size
    for (mask in 0 until (1 shl n)) {
        val s = mutableSetOf<String>()
        for (i in 0 until n) if ((mask shr i) and 1 == 1) s += items[i]
        out += s
    }
    return out
}

fun ablationSweep(name: String, corpus: Map<String, PhaseAResult>) =
    corpus.flatMap { (fixture, phaseA) ->
        val baseline = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
        val baselineRanking = rankingOf(baseline)
        val baselineMag = baseline.divergencePoints.sumOf { kotlin.math.abs(it.magnitude) }
        powerSet(PROC_TERMS).map { active ->
            val perturbed = ReportAssembler.assemble(phaseA, ablate(active))
            val pr = rankingOf(perturbed)
            val perturbedMag = perturbed.divergencePoints.sumOf { kotlin.math.abs(it.magnitude) }
            mapOf(
                "set" to name, "fixture" to fixture,
                "variant" to if (active.isEmpty()) "(none)" else active.toSortedSet().joinToString("+"),
                "active_count" to active.size,
                "tau" to kendallTauB(baselineRanking, pr),
                "perturbed_mag" to perturbedMag,
                "baseline_mag" to baselineMag,
            )
        }
    }.toDataFrame()

val ablInj   = ablationSweep("Injection (45)",  injection)
val ablUser  = ablationSweep("User study (12)", userStudy)
val ablAgent = ablationSweep("Agent (6)",       agent)

Monotone-by-active-count on the injection set (reproduces Table 5.4).

In [12]:
ablInj.groupBy("active_count").aggregate {
    mean { "tau"<Double>() } into "mean_tau"
    val sumPert = sum { "perturbed_mag"<Double>() }
    val sumBase = sum { "baseline_mag"<Double>() }
    (if (sumBase > 0.0) sumPert / sumBase else 0.0) into "sum_over_sum"
    count() into "n"
}.sortBy("active_count")

active_count,mean_tau,sum_over_sum,n
0,0.744074,0.000000,45
1,0.743439,0.170361,315
2,0.746420,0.326902,945
3,0.753524,0.473481,1575
4,0.765259,0.612479,1575
5,0.782134,0.745484,945
6,0.804656,0.873974,315
7,0.833333,1.000000,45


Per-term single-knob recovery on the injection set (reproduces Table 5.5): each
process-side term active alone, the other five zeroed. Columns: mean $\tau$ against
production ranking, mean per-fixture recovery, and sum-over-sum recovery.

In [13]:
fun soloRow(df: AnyFrame, term: String): Map<String, Any> {
    val rows = df.filter { "variant"<String>() == term }
    val meanTau = rows["tau"].cast<Double>().mean()
    // mean per-fixture recovery: |perturbed_mag| / |baseline_mag|, averaged
    val perFixture = (0 until rows.rowsCount()).map { i ->
        val b = rows["baseline_mag"][i] as Double
        val p = rows["perturbed_mag"][i] as Double
        if (b > 0.0) p / b else 1.0
    }
    val meanRec = perFixture.average()
    val sumPert = rows["perturbed_mag"].cast<Double>().sum()
    val sumBase = rows["baseline_mag"].cast<Double>().sum()
    val sumOverSum = if (sumBase > 0.0) sumPert / sumBase else 0.0
    return mapOf(
        "term" to term,
        "mean τ" to "%.3f".format(meanTau),
        "mean recovery" to "%.3f".format(meanRec),
        "sum/sum" to "%.3f".format(sumOverSum),
    )
}

PROC_TERMS.map { soloRow(ablInj, it) }
    .sortedByDescending { (it["sum/sum"] as String).toDouble() }
    .toDataFrame()

term,mean τ,mean recovery,sum/sum
length,0.744,0.578,0.328
broken,0.789,0.553,0.302
manualIde,0.744,0.442,0.261
lag,0.669,0.423,0.132
skipTests,0.718,0.398,0.089
commitGap,0.796,0.368,0.080
gain,0.744,0.311,0.000


Per-term leave-one-out recovery on the injection set (reproduces Table 5.6): five
process-side terms active, one removed. Removing $W_{\textsc{l}}$ should produce the
largest sum/sum drop; removing $W_{\textsc{g}}$ should push sum/sum above 1.0 (the
regulariser-against-penalties effect discussed in the chapter).

In [14]:
fun looRow(df: AnyFrame, removed: String): Map<String, Any> {
    val target = (allTerms - removed).toSortedSet()
    val rows = df.filter { "variant"<String>().split("+").toSortedSet() == target }
    val meanTau = rows["tau"].cast<Double>().mean()
    val perFixture = (0 until rows.rowsCount()).map { i ->
        val b = rows["baseline_mag"][i] as Double
        val p = rows["perturbed_mag"][i] as Double
        if (b > 0.0) p / b else 1.0
    }
    val meanRec = perFixture.average()
    val sumPert = rows["perturbed_mag"].cast<Double>().sum()
    val sumBase = rows["baseline_mag"].cast<Double>().sum()
    val sumOverSum = if (sumBase > 0.0) sumPert / sumBase else 0.0
    return mapOf(
        "removed term" to removed,
        "mean τ" to "%.3f".format(meanTau),
        "mean recovery" to "%.3f".format(meanRec),
        "sum/sum" to "%.3f".format(sumOverSum),
    )
}

PROC_TERMS.map { looRow(ablInj, it) }
    .sortedBy { (it["sum/sum"] as String).toDouble() }
    .toDataFrame()

removed term,mean τ,mean recovery,sum/sum
length,0.833,0.817,0.718
manualIde,0.833,0.911,0.822
broken,0.731,0.966,0.833
lag,0.803,0.880,0.842
skipTests,0.815,0.903,0.899
commitGap,0.791,0.956,0.943
gain,0.826,0.991,1.060


Cross-set leave-one-out τ (reproduces Table 5.7).

In [15]:
fun looTau(df: AnyFrame, removed: String): Double {
    val target = (allTerms - removed).toSortedSet()
    val rows = df.filter { "variant"<String>().split("+").toSortedSet() == target }
    return rows["tau"].cast<Double>().mean()
}

PROC_TERMS.sorted().map { removed ->
    mapOf(
        "removed" to removed,
        "injection τ" to "%.3f".format(looTau(ablInj, removed)),
        "user-study τ" to "%.3f".format(looTau(ablUser, removed)),
    )
}.toDataFrame()

removed,injection τ,user-study τ
broken,0.731,0.689
commitGap,0.791,0.713
gain,0.826,0.736
lag,0.803,0.820
length,0.833,0.606
manualIde,0.833,0.657
skipTests,0.815,0.719


## §5.2 Testing the detector

### §5.2.1 Inter-rater κ

Loads the three rater manifests, computes pair-wise Cohen's κ per kind across the 45
sessions (reproduces Table 5.8).

In [16]:
fun parseCsvRow(line: String): List<String> {
    val out = mutableListOf<String>()
    val cur = StringBuilder()
    var inQuotes = false; var i = 0
    while (i < line.length) {
        val c = line[i]
        when {
            inQuotes && c == '"' && i + 1 < line.length && line[i + 1] == '"' -> { cur.append('"'); i += 2; continue }
            inQuotes && c == '"' -> inQuotes = false
            !inQuotes && c == '"' -> inQuotes = true
            !inQuotes && c == ',' -> { out += cur.toString(); cur.setLength(0) }
            else -> cur.append(c)
        }
        i++
    }
    out += cur.toString()
    return out
}

fun loadManifest(path: String): Map<String, Set<String>> {
    val lines = File(repoRoot, path).readLines()
    val header = parseCsvRow(lines[0])
    val sid = header.indexOf("session_id")
    val kinds = header.indexOf("expected_kinds")
    return lines.drop(1).filter { it.isNotBlank() }
        .associate {
            val row = parseCsvRow(it)
            row[sid] to row[kinds].split(";").mapNotNull { k -> k.trim().ifEmpty { null } }.toSet()
        }
}

// Cohen's κ: inter-rater agreement on a binary judgement,
// adjusted for chance. κ = (po − pe) / (1 − pe), where po is the
// fraction of items the two raters agreed on, and pe is the
// agreement expected if both rated independently with their own
// marginals. Landis-Koch interpretation: <0.20 slight, 0.21-0.40
// fair, 0.41-0.60 moderate, 0.61-0.80 substantial, 0.81-1.00 almost
// perfect. NaN when both raters labelled every item the same way
// (denominator collapses).
fun cohenKappa(yy: Int, yn: Int, ny: Int, nn: Int): Double {
    val n = yy + yn + ny + nn
    if (n == 0) return Double.NaN
    val po = (yy + nn).toDouble() / n
    val r1y = (yy + yn).toDouble() / n
    val r2y = (yy + ny).toDouble() / n
    val pe = r1y * r2y + (1 - r1y) * (1 - r2y)
    return if (pe == 1.0) if (po == 1.0) 1.0 else Double.NaN else (po - pe) / (1 - pe)
}

val raters = mapOf(
    "Author" to loadManifest("fixtures/sessions/manifest-v2.csv"),
    "P1"     to loadManifest("fixtures/sessions/manifest-v2-will.csv"),
    "P2"     to loadManifest("fixtures/sessions/manifest-v2-yukie.csv"),
)
val pairs = listOf("Author" to "P1", "Author" to "P2", "P1" to "P2")
val kinds = listOf("ORDERING", "IDE_REPLAY", "REWORK", "HYGIENE")

kinds.map { kind ->
    val row = mutableMapOf<String, Any>("kind" to kind)
    for ((a, b) in pairs) {
        val ax = raters[a]!!; val bx = raters[b]!!
        val common = ax.keys intersect bx.keys
        var yy = 0; var yn = 0; var ny = 0; var nn = 0
        for (sid in common) {
            val ai = kind in (ax[sid] ?: emptySet())
            val bi = kind in (bx[sid] ?: emptySet())
            when { ai && bi -> yy++; ai && !bi -> yn++; !ai && bi -> ny++; else -> nn++ }
        }
        row["$a vs $b"] = "%.3f".format(cohenKappa(yy, yn, ny, nn))
    }
    row
}.toDataFrame()

kind,Author vs P1,Author vs P2,P1 vs P2
ORDERING,1.000,1.000,1.000
IDE_REPLAY,1.000,1.000,1.000
REWORK,0.861,0.861,1.000
HYGIENE,0.848,0.723,0.862


### §5.2.2 Divergence experiment

Per-kind precision and recall against the author manifest, on the 45-session injection set.

In [17]:
// Decode each session's report and check against the manifest's expected_kinds.
val authorManifest = raters["Author"]!!

val divRows = injection.map { (sid, phaseA) ->
    val report = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
    val observed = report.divergencePoints.groupBy { it.kind.name }.mapValues { it.value.size }
    val expected = authorManifest[sid] ?: emptySet()
    mapOf(
        "session_id" to sid,
        "expected" to expected.sorted().joinToString(";"),
    ) + kinds.associateWith { k -> observed[k] ?: 0 }
}.toDataFrame()
divRows.head(5)

session_id,expected,ORDERING,IDE_REPLAY,REWORK,HYGIENE
001,IDE_REPLAY;ORDERING,0,1,0,0
002,IDE_REPLAY;ORDERING,0,1,0,0
003,IDE_REPLAY;ORDERING,0,1,0,0
004,IDE_REPLAY;ORDERING,0,1,0,0
005,IDE_REPLAY;ORDERING,0,1,0,0


Precision / recall per kind (reproduces Table 5.9).

In [18]:
kinds.map { kind ->
    var tp = 0; var fp = 0; var fn = 0
    for (i in 0 until divRows.rowsCount()) {
        val row = divRows[i]
        val isExp = kind in (row["expected"] as String).split(";").mapNotNull { it.trim().ifEmpty { null } }.toSet()
        val obs = (row[kind] as Int) > 0
        when {
            isExp && obs -> tp++
            isExp && !obs -> fn++
            !isExp && obs -> fp++
        }
    }
    val p = if (tp + fp > 0) tp.toDouble() / (tp + fp) else Double.NaN
    val r = if (tp + fn > 0) tp.toDouble() / (tp + fn) else Double.NaN
    mapOf(
        "kind" to kind, "TP" to tp, "FP" to fp, "FN" to fn,
        "precision" to "%.3f".format(p),
        "recall (any-expected)" to "%.3f".format(r),
    )
}.toDataFrame()

kind,TP,FP,FN,precision,recall (any-expected)
ORDERING,14,0,24,1.000,0.368
IDE_REPLAY,16,0,5,1.000,0.762
REWORK,9,0,0,1.000,1.000
HYGIENE,8,0,0,1.000,1.000


Detection quality (reproduces Table 5.10) — divergence points fired per kind and the
fraction whose best alternative beats the user's trajectory under production weights.

In [19]:
kinds.map { kind ->
    val divKind = com.github.ethanhosier.analysis.pipeline.DivergenceKind.valueOf(kind)
    var fires = 0
    var beats = 0
    for ((_, phaseA) in injection) {
        val report = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
        for (dp in report.divergencePoints) {
            if (dp.kind == divKind) {
                fires += 1
                if (dp.magnitude > 0.0) beats += 1
            }
        }
    }
    val frac = if (fires > 0) beats.toDouble() / fires else Double.NaN
    mapOf(
        "kind" to kind,
        "DPs fired" to fires,
        "beats user" to beats,
        "beats fraction" to "%.3f".format(frac),
    )
}.toDataFrame()

kind,DPs fired,beats user,beats fraction
ORDERING,24,10,0.417
IDE_REPLAY,16,12,0.750
REWORK,13,6,0.462
HYGIENE,13,13,1.000


Injection prominence (reproduces Table 5.11) — the playbook's `pattern` column maps each
session to one deliberately-injected divergence kind; for each session where the detector caught
that kind with a positive-magnitude alternative, record the rank of the highest-magnitude DP of
the injection kind within the full sorted ranking. The chapter's headline claim is that every
caught injection ranks at top-1.

In [20]:
// Pattern -> injection kind, matching INJECTION_KIND_BY_PATTERN in
// DivergenceExperiment.kt:176. Controls map to null and are skipped.
val INJECTION_KIND_BY_PATTERN: Map<String, String?> = mapOf(
    "ManualExtractMethod"  to "IDE_REPLAY",
    "ManualRenameMethod"   to "IDE_REPLAY",
    "ManualInlineMethod"   to "IDE_REPLAY",
    "ManualMoveMethod"     to "IDE_REPLAY",
    "ManualExtractVariable" to "IDE_REPLAY",
    "SuboptimalOrdering"   to "ORDERING",
    "AddThenRevert"        to "REWORK",
    "SkippedTests"         to "HYGIENE",
    "NoCommitStretch"      to "HYGIENE",
    "Control"              to null,
)

// Load the pattern column alongside expected_kinds.
fun loadPatterns(path: String): Map<String, String> {
    val lines = File(repoRoot, path).readLines()
    val header = parseCsvRow(lines[0])
    val sid = header.indexOf("session_id")
    val pat = header.indexOf("pattern")
    return lines.drop(1).filter { it.isNotBlank() }
        .associate {
            val row = parseCsvRow(it)
            row[sid] to row[pat]
        }
}
val patternsBySid = loadPatterns("fixtures/sessions/manifest-v2.csv")

// Build the ranking once per session, then look up the injection kind's
// top rank. magnitude > 0 + matching kind = "caught"; rank uses the same
// tie-broken-by-stepIndex order as the rest of the chapter.
fun rankedDps(report: AnalysisReport) =
    report.divergencePoints.sortedWith(
        compareByDescending<com.github.ethanhosier.analysis.pipeline.DivergencePoint> { it.magnitude }
            .thenBy { it.stepIndex }
    )

val prominenceByKind = mutableMapOf<String, MutableList<Int>>()
for ((sid, phaseA) in injection) {
    val pattern = patternsBySid[sid] ?: continue
    val injKind = INJECTION_KIND_BY_PATTERN[pattern] ?: continue
    val report = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
    val ranked = rankedDps(report)
    val injKindEnum = com.github.ethanhosier.analysis.pipeline.DivergenceKind.valueOf(injKind)
    val firstHitRank = ranked.indexOfFirst { it.kind == injKindEnum && it.magnitude > 0.0 } + 1
    if (firstHitRank > 0) {
        prominenceByKind.getOrPut(injKind) { mutableListOf() }.add(firstHitRank)
    }
}

kinds.map { kind ->
    val ranks = prominenceByKind[kind] ?: emptyList()
    mapOf(
        "kind" to kind,
        "caught" to ranks.size,
        "mean rank" to (if (ranks.isNotEmpty()) "%.2f".format(ranks.average()) else "—"),
        "all at rank 1?" to (if (ranks.isNotEmpty()) (ranks.all { it == 1 }).toString() else "—"),
    )
}.toDataFrame()

kind,caught,mean rank,all at rank 1?
ORDERING,1,1.00,true
IDE_REPLAY,12,1.00,true
REWORK,5,1.00,true
HYGIENE,8,1.13,false


Best-alt magnitude range across the injection sessions — gives an absolute scale for the
per-kind beat fractions above. Cited in the detection-quality paragraph of §5.2.2.

In [21]:
val bestAlts = injection.values.mapNotNull { phaseA ->
    ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
        .divergencePoints.maxOfOrNull { it.magnitude }
}
mapOf(
    "n sessions with any DP" to bestAlts.size,
    "mean best-alt magnitude" to "%.2f".format(bestAlts.average()),
    "min" to bestAlts.min().toInt(),
    "max" to bestAlts.max().toInt(),
)

{n sessions with any DP=36, mean best-alt magnitude=7.42, min=-5, max=24}

## §5.3 User and agent studies

### Per-session × kind distribution

Strict `magnitude > 0` filter, matching the §3.4 ORDERING-detection paragraph. The
`userRows` dataframe printed at the end of this cell reproduces Table 5.12.

In [22]:
fun perSessionRow(sid: String, phaseA: PhaseAResult): Map<String, Any> {
    val report = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
    val counts = kinds.associateWith { k ->
        report.divergencePoints.count { it.kind.name == k && it.magnitude > 0.0 }
    }
    return mapOf("session" to sid) + counts + ("total" to counts.values.sum())
}

val userRows  = userStudy.map { (k, v) -> perSessionRow(k, v) }.toDataFrame()
val agentRows = agent.map { (k, v) -> perSessionRow(k, v) }.toDataFrame()
userRows

session,ORDERING,IDE_REPLAY,REWORK,HYGIENE,total
alex-baseline-01,0,0,0,0,0
alex-baseline-02,0,3,1,5,9
alex-baseline-03,0,8,1,4,13
alex-baseline-04,0,0,0,0,0
alex-baseline-05,0,5,0,4,9
alex-baseline-06,0,5,1,3,9
bobby-01,0,0,0,0,0
bobby-02,0,2,0,4,6
bobby-03,0,1,0,0,1
bobby-04,0,0,0,0,0


Per-kind trajectory across the six-session arc, summed across P1 + P2 (reproduces Table 5.13).

In [23]:
val combinedTraj = (1..6).map { idx ->
    val matching = (0 until userRows.rowsCount())
        .map { userRows[it] }
        .filter { (it["session"] as String).endsWith("-%02d".format(idx)) }
    val totals = kinds.associateWith { k -> matching.sumOf { (it[k] as Int) } }
    mapOf("session" to "S$idx") + totals + ("total" to totals.values.sum())
}.toDataFrame()
combinedTraj

session,ORDERING,IDE_REPLAY,REWORK,HYGIENE,total
S1,2,13,0,7,22
S2,0,9,1,22,32
S3,1,17,3,9,30
S4,0,0,0,0,0
S5,0,12,0,8,20
S6,2,9,1,5,17


In [24]:
val trajLong = kinds.flatMap { k ->
    (0 until combinedTraj.rowsCount()).map { i ->
        mapOf("kind" to k, "session" to combinedTraj["session"][i] as String, "count" to combinedTraj[k][i] as Int)
    }
}.toDataFrame()

trajLong.plot {
    line   { x("session"); y("count"); color("kind") }
    points { x("session"); y("count"); color("kind") }
    layout { title = "Per-kind trajectory (P1 + P2 combined)" }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="sTMAj4" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("sTMAj4");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"Per-kind trajectory (P1 + P2 combined)"
},
"mapping":{
},
"data":{
"session":["S1","S2","S3","S4","S5","S6","S1","S2","S3","S4","S5","S6","S1","S2","S3","S4","S5","S6","S1","S2","S3","S4","S5","S6"],
"kind":["ORDERING","ORDERING","ORDERING","ORDERING","ORDERING","ORDERING","IDE_REPLAY","IDE_REPLAY","IDE_REPLAY","IDE_REPLAY","IDE_REPLAY","IDE_REPLAY","REWORK","REWORK","REWORK","REWORK","REWORK","REWORK","HYGIENE","HYGIENE","HYGIENE","HYGIENE","HYGIENE","HYGIENE"],
"count":[2.0,0.0,1.0,0.0,0.0,2.0,13.0,9.0,17.0,0.0,12.0,9.0,0.0,1.0,3.0,0.0,0.0,1.0,7.0,22.0,9.0,0.0,8.0,5.0]
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"color",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"session",
"y":"count",
"color":"kind"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"line",
"data":{
}
},{
"mapping":{
"x":"session",
"y":"count",
"color":"kind"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data":{
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"session"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"kind"
}]
},
"spec_id":"5"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 S1 
 
 
 
 
 
 
 
 
 S2 
 
 
 
 
 
 
 
 
 S3 
 
 
 
 
 
 
 
 
 S4 
 
 
 
 
 
 
 
 
 S5 
 
 
 
 
 
 
 
 
 S6 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 5 
 
 
 
 
 
 
 10 
 
 
 
 
 
 
 15 
 
 
 
 
 
 
 20 
 
 
 
 
 
 
 
 
 Per-kind trajectory (P1 + P2 combined) 
 
 
 
 
 count 
 
 
 
 
 session 
 
 
 
 
 
 
 
 
 kind 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 ORDERING 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 IDE_REPLAY 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 REWORK 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 HYGIENE

### §5.3.2 Agent extension corpus (per-session DP counts)

In [25]:
agentRows

session,ORDERING,IDE_REPLAY,REWORK,HYGIENE,total
claude-2-opus-4.7-claude-code-01,0,0,0,0,0
claude-2-opus-4.7-claude-code-02,0,0,0,0,0
claude-2-opus-4.7-claude-code-03,0,1,0,0,1
claude-2-opus-4.7-claude-code-04,0,0,0,0,0
claude-2-opus-4.7-claude-code-05,0,2,0,0,2
claude-2-opus-4.7-claude-code-06,0,2,0,0,2
claude-4.7-opus-medium-claude-code-ba...,0,0,0,0,0
claude-4.7-opus-medium-claude-code-ba...,0,0,0,0,0
claude-4.7-opus-medium-claude-code-ba...,0,1,0,0,1
claude-4.7-opus-medium-claude-code-ba...,0,0,0,0,0


Arm-classification and per-participant/per-cell helpers used by the rest of this
section. The `-baseline` suffix on the participant or stack stem marks the no-feedback
arm; the trailing `-NN` of every session id is the in-arc session index 1..6.

In [26]:
// `-baseline` suffix on the participant (user-sessions) or stack (agent-sessions)
// stem marks the no-feedback control arm.
fun isBaseline(sid: String): Boolean =
    sid.substringBeforeLast('-').endsWith("-baseline")

// Strip the trailing -NN to get participant or cell name.
fun participantOf(sid: String): String = sid.substringBeforeLast('-')
fun cellOf(sid: String): String = sid.substringBeforeLast('-')

// Extract the trailing 1..6 in-arc index.
fun sessionIdx(sid: String): Int? =
    sid.substringAfterLast('-').toIntOrNull()

// Pretty arm labels. P1..P5 anonymise the user-study participants; agent cells are
// reported by their model x harness names.
val userPretty = mapOf(
    "will" to "P1", "yukie" to "P2", "bobby" to "P3",
    "alex-baseline" to "P4", "vlad-baseline" to "P5",
)
// Accepts either a session id or a participant key (e.g. "will" or "will-01").
fun prettyUser(s: String): String {
    val key = if (sessionIdx(s) != null) participantOf(s) else s
    return userPretty[key] ?: key
}

Per-arm trajectory aggregator. Given a corpus (`userStudy` or `agent`) and a
`group` function (e.g. `participantOf` for users, `cellOf` for agent stacks), produces a
DataFrame of per-group DP counts at S1..S6 plus a total column. The arm-aware variant
groups by arm (`with-feedback` / `baseline`) by summing across the groups that share an
arm.

In [27]:
// Per-group arc of DP totals. df is `userRows` or `agentRows` (one row per session).
fun arcByGroup(df: AnyFrame, group: (String) -> String): List<Map<String, Any>> {
    val arcs = mutableMapOf<String, IntArray>()
    for (i in 0 until df.rowsCount()) {
        val sid = df["session"][i] as String
        val idx = sessionIdx(sid) ?: continue
        val g = group(sid)
        val arr = arcs.getOrPut(g) { IntArray(6) }
        arr[idx - 1] += df["total"][i] as Int
    }
    return arcs.toSortedMap().map { (g, arr) ->
        mapOf<String, Any>("group" to g) +
            (1..6).associate { "S$it" to arr[it - 1] } +
            ("Δ" to (arr[5] - arr[0])) +
            ("Σ" to arr.sum())
    }
}

val userArcByPpt  = arcByGroup(userRows,  ::participantOf).toDataFrame()
val agentArcByCell = arcByGroup(agentRows, ::cellOf).toDataFrame()
userArcByPpt

group,S1,S2,S3,S4,S5,S6,Δ,Σ
alex-baseline,0,9,13,0,9,9,9,40
bobby,0,6,1,0,3,0,0,10
vlad-baseline,10,6,6,0,4,6,-4,32
will,7,5,6,0,2,1,-6,21
yukie,5,6,4,0,2,1,-4,18


In [28]:
agentArcByCell

group,S1,S2,S3,S4,S5,S6,Δ,Σ
claude-2-opus-4.7-claude-code,0,0,1,0,2,2,2,5
claude-4.7-opus-medium-claude-code-ba...,0,0,1,0,1,1,1,3
claude-opus-4.7-medium-opencode,0,0,1,0,1,2,2,4
claude-opus-4.7-thinking-medium-curso...,0,0,2,0,2,2,2,6
gemini-3.5-flash-medium-opencode,3,0,2,0,2,2,-1,9
gpt-5.5-medium-cursor-cli,3,0,2,0,1,1,-2,7
gpt-5.5-medium-opencode-baseline,1,0,1,0,1,1,0,4
openai-gpt-5.5-medium-opencode,1,0,2,0,1,1,0,5


Per-arm DP trajectory summed across all participants (user-study) or all cells
(agent extension). Reproduces the arm rows of the chapter's trajectory tables.

In [29]:
fun armArc(df: AnyFrame): List<Map<String, Any>> {
    val arcs = mapOf("with-feedback" to IntArray(6), "baseline" to IntArray(6))
    for (i in 0 until df.rowsCount()) {
        val sid = df["session"][i] as String
        val idx = sessionIdx(sid) ?: continue
        val arm = if (isBaseline(sid)) "baseline" else "with-feedback"
        arcs[arm]!![idx - 1] += df["total"][i] as Int
    }
    return arcs.map { (arm, arr) ->
        mapOf<String, Any>("arm" to arm) +
            (1..6).associate { "S$it" to arr[it - 1] } +
            ("Δ" to (arr[5] - arr[0])) +
            ("Σ" to arr.sum())
    }
}

armArc(userRows).toDataFrame()

arm,S1,S2,S3,S4,S5,S6,Δ,Σ
with-feedback,12,17,11,0,7,2,-10,49
baseline,10,15,19,0,13,15,5,72


In [30]:
armArc(agentRows).toDataFrame()

arm,S1,S2,S3,S4,S5,S6,Δ,Σ
with-feedback,7,0,10,0,9,10,3,36
baseline,1,0,2,0,2,2,1,7


### Process score trajectory: gain-stripped weights

For each session, the trajectory-final process score is computed under a copy of the
production weighting with the cleanliness-gain weight `W_g` set to zero. The
gain-stripped view isolates the process-discipline terms (broken-time, skipped-tests,
manual-when-IDE, length bonus, commit-gap) from the cleanliness-gain term, mirroring
the §5.1.2 ablation finding that `W_g` acts as a regulariser on divergence-point
ranking. Reproduces Table 5.14 (user-study) and Table 5.16 (agent extension).

In [31]:
// Per-group arc of trajectory-final process scores under the given config.
fun arcScoresBy(
    sessions: Map<String, PhaseAResult>,
    group: (String) -> String,
    cfg: ScoringConfig,
): Map<String, IntArray> {
    val out = mutableMapOf<String, IntArray>()
    for ((sid, pa) in sessions) {
        val idx = sessionIdx(sid) ?: continue
        val g = group(sid)
        val arr = out.getOrPut(g) { IntArray(6) { -1 } }
        val score = ReportAssembler.assemble(pa, cfg).checkpoints.last()
            .derivedMetrics?.process?.total ?: -1
        arr[idx - 1] = score
    }
    return out.toSortedMap()
}

// Re-derive a gain-stripped config in-memory so the notebook is self-contained.
val gainZero = ScoringConfig.PRODUCTION.copy(
    process = ScoringConfig.PRODUCTION.process.copy(gain = 0.0),
)

fun scoreRows(
    scores: Map<String, IntArray>,
    armOf: (String) -> String,
    pretty: (String) -> String,
): List<Map<String, Any>> = scores.toSortedMap().map { (g, arr) ->
    val s1 = arr[0]; val s6 = arr[5]
    mapOf<String, Any>("arm" to armOf(g), "group" to pretty(g)) +
        (1..6).associate { "S$it" to arr[it - 1] } +
        ("ΔJ" to (s6 - s1)) +
        ("slope" to "%.2f".format((s6 - s1) / 5.0))
}

val userScores  = arcScoresBy(userStudy, ::participantOf, gainZero)
val agentScores = arcScoresBy(agent,     ::cellOf,        gainZero)

scoreRows(
    userScores,
    armOf = { if (it.endsWith("-baseline")) "baseline" else "with-feedback" },
    pretty = ::prettyUser,
).toDataFrame()

arm,group,S1,S2,S3,S4,S5,S6,ΔJ,slope
baseline,P4,22,17,8,22,12,15,-7,-1.40
with-feedback,P3,39,15,42,42,31,48,9,1.80
baseline,P5,23,17,9,36,11,20,-3,-0.60
with-feedback,P1,25,22,41,38,34,48,23,4.60
with-feedback,P2,16,13,29,39,37,47,31,6.20


In [32]:
scoreRows(
    agentScores,
    armOf = { if (it.endsWith("-baseline")) "baseline" else "with-feedback" },
    pretty = { it },
).toDataFrame()

arm,group,S1,S2,S3,S4,S5,S6,ΔJ,slope
with-feedback,claude-2-opus-4.7-claude-code,35,39,40,45,40,35,0,0.00
baseline,claude-4.7-opus-medium-claude-code-ba...,36,34,39,45,36,40,4,0.80
with-feedback,claude-opus-4.7-medium-opencode,37,36,40,45,37,38,1,0.20
with-feedback,claude-opus-4.7-thinking-medium-curso...,35,39,40,45,39,39,4,0.80
with-feedback,gemini-3.5-flash-medium-opencode,33,37,40,45,39,37,4,0.80
with-feedback,gpt-5.5-medium-cursor-cli,23,38,39,45,37,37,14,2.80
baseline,gpt-5.5-medium-opencode-baseline,25,34,40,45,36,38,13,2.60
with-feedback,openai-gpt-5.5-medium-opencode,35,38,40,45,36,36,1,0.20


### Per-session broken-build percentage by participant

Broken wall-clock time divided by total session wall-clock time, parsed from the
final checkpoint's `process.contributions[broken].detail` string. These percentages
surface on the dashboard as the `BUILD_OFTEN_BROKEN` advice item when they cross the
$15\%$ warning threshold or the $30\%$ critical threshold. Reproduces the per-session
broken-build numbers cited in the §5.4 cell-1 Thread 1.

In [33]:
// Parse details like "1m04s of 4m44s broken (23%) - 4 of 10 checkpoint".
val brokenPctRe = Regex("\\((\\d+)%\\)")

fun brokenPctBy(
    sessions: Map<String, PhaseAResult>,
    group: (String) -> String,
): Map<String, IntArray> {
    val out = mutableMapOf<String, IntArray>()
    for ((sid, pa) in sessions) {
        val idx = sessionIdx(sid) ?: continue
        val g = group(sid)
        val arr = out.getOrPut(g) { IntArray(6) { -1 } }
        val report = ReportAssembler.assemble(pa, ScoringConfig.PRODUCTION)
        val last = report.checkpoints.lastOrNull() ?: continue
        val contrib = last.derivedMetrics?.process?.contributions?.firstOrNull { it.id == "broken" }
        val detail = contrib?.detail ?: ""
        arr[idx - 1] = brokenPctRe.find(detail)?.groupValues?.get(1)?.toIntOrNull() ?: 0
    }
    return out.toSortedMap()
}

fun brokenRows(
    pcts: Map<String, IntArray>,
    pretty: (String) -> String,
): List<Map<String, Any>> = pcts.map { (g, arr) ->
    val present = arr.filter { it >= 0 }
    val mean = if (present.isNotEmpty()) present.average() else 0.0
    val peak = present.maxOrNull() ?: 0
    mapOf<String, Any>("actor" to pretty(g)) +
        (1..6).associate { "S$it" to "${arr[it - 1]}%" } +
        ("mean" to "%.1f%%".format(mean)) +
        ("peak" to "$peak%")
}

brokenRows(brokenPctBy(userStudy, ::participantOf), ::prettyUser).toDataFrame()

actor,S1,S2,S3,S4,S5,S6,mean,peak
P4,10%,20%,65%,62%,45%,39%,40.2%,65%
P3,7%,2%,15%,3%,16%,0%,7.2%,16%
P5,23%,40%,63%,20%,49%,28%,37.2%,63%
P1,23%,0%,3%,12%,6%,0%,7.3%,23%
P2,52%,0%,0%,10%,22%,0%,14.0%,52%


In [34]:
brokenRows(brokenPctBy(agent, ::cellOf), { it }).toDataFrame()

actor,S1,S2,S3,S4,S5,S6,mean,peak
claude-2-opus-4.7-claude-code,22%,0%,0%,0%,0%,16%,6.3%,22%
claude-4.7-opus-medium-claude-code-ba...,17%,0%,0%,0%,0%,0%,2.8%,17%
claude-opus-4.7-medium-opencode,13%,0%,0%,0%,0%,5%,3.0%,13%
claude-opus-4.7-thinking-medium-curso...,19%,0%,0%,0%,0%,0%,3.2%,19%
gemini-3.5-flash-medium-opencode,3%,0%,0%,0%,0%,10%,2.2%,10%
gpt-5.5-medium-cursor-cli,32%,0%,0%,0%,0%,0%,5.3%,32%
gpt-5.5-medium-opencode-baseline,30%,0%,0%,0%,0%,0%,5.0%,30%
openai-gpt-5.5-medium-opencode,0%,0%,0%,0%,0%,5%,0.8%,5%


### Corpus-expansion findings dump

Appended cell: emits per-session production J, gain-stripped J, and DP counts by kind for every user + agent session as plain TSV so downstream tools (e.g. the corpus-expansion comparison report) get the full corpus without dataframe-display truncation.

In [35]:
val expansionGainZero = ScoringConfig.PRODUCTION.copy(
    process = ScoringConfig.PRODUCTION.process.copy(gain = 0.0),
)

fun emitDump(label: String, sessions: Map<String, PhaseAResult>) {
    println("### $label ###")
    println("session\tORDERING\tIDE_REPLAY\tREWORK\tHYGIENE\ttotal\tJ_prod\tJ_gain0")
    for ((sid, phaseA) in sessions.toSortedMap()) {
        val prod = ReportAssembler.assemble(phaseA, ScoringConfig.PRODUCTION)
        val g0   = ReportAssembler.assemble(phaseA, expansionGainZero)
        val counts = kinds.associateWith { k -> prod.divergencePoints.count { it.kind.name == k && it.magnitude > 0.0 } }
        val jProd = prod.checkpoints.lastOrNull()?.derivedMetrics?.process?.total ?: -1
        val jG0   = g0  .checkpoints.lastOrNull()?.derivedMetrics?.process?.total ?: -1
        println("$sid\t${counts["ORDERING"]}\t${counts["IDE_REPLAY"]}\t${counts["REWORK"]}\t${counts["HYGIENE"]}\t${counts.values.sum()}\t$jProd\t$jG0")
    }
    println()
}

emitDump("USER-STUDY", userStudy)
emitDump("AGENT", agent)

### USER-STUDY ###
session	ORDERING	IDE_REPLAY	REWORK	HYGIENE	total	J_prod	J_gain0


alex-baseline-01	0	0	0	0	0	0	22
alex-baseline-02	0	3	1	5	9	26	17
alex-baseline-03	0	8	1	4	13	10	8


alex-baseline-04	0	0	0	0	0	72	22
alex-baseline-05	0	5	0	4	9	41	12
alex-baseline-06	0	5	1	3	9	27	15


bobby-01	0	0	0	0	0	0	39
bobby-02	0	2	0	4	6	65	15
bobby-03	0	1	0	0	1	42	42
bobby-04	0	0	0	0	0	92	42


bobby-05	0	2	0	1	3	9	31
bobby-06	0	0	0	0	0	53	48
vlad-baseline-01	1	5	0	4	10	23	23


...

gpt-5.5-medium-opencode-baseline-05	0	1	0	0	1	86	36


gpt-5.5-medium-opencode-baseline-06	0	1	0	0	1	68	38


openai-gpt-5.5-medium-opencode-01	0	1	0	0	1	85	35
openai-gpt-5.5-medium-opencode-02	0	0	0	0	0	63	38


openai-gpt-5.5-medium-opencode-03	0	2	0	0	2	30	40
openai-gpt-5.5-medium-opencode-04	0	0	0	0	0	95	45


openai-gpt-5.5-medium-opencode-05	0	1	0	0	1	86	36
openai-gpt-5.5-medium-opencode-06	0	1	0	0	1	59	36
